# Customer Churn Prediction
**RetailPulse | MLOps + Business Analytics**

Binary classifier to predict which customers will churn (no purchase in 90+ days), enabling proactive retention campaigns.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (roc_auc_score, f1_score, classification_report,
                              ConfusionMatrixDisplay, RocCurveDisplay)
from sklearn.preprocessing import StandardScaler
import mlflow
import mlflow.sklearn
import psycopg2

mlflow.set_tracking_uri('http://mlflow:5000')
conn = psycopg2.connect(
    host='postgres', dbname='retailpulse',
    user='retailpulse', password='retailpulse123'
)

In [ ]:
# Load churn feature table from Gold layer
df = pd.read_sql("""
    SELECT days_since_last_order, total_orders, total_revenue, avg_order_value,
           cancellation_rate, r_score, f_score, m_score, rfm_total,
           avg_session_duration, avg_products_viewed, total_sessions_30d,
           label
    FROM gold.agg_churn_features
    WHERE label IS NOT NULL
""", conn)

df = df.fillna(0)
print(f'Dataset: {len(df):,} customers')
print(f'Churn rate: {df["label"].mean()*100:.1f}%')
df.head()

In [ ]:
# EDA: Feature distributions by churn label
key_features = ['days_since_last_order', 'total_orders', 'rfm_total', 'total_sessions_30d']

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, feat in zip(axes.flatten(), key_features):
    for label, color, name in [(0, '#4878CF', 'Retained'), (1, '#E24A33', 'Churned')]:
        subset = df[df['label'] == label][feat]
        ax.hist(subset.clip(0, subset.quantile(0.99)), bins=30,
                alpha=0.6, color=color, label=name, density=True)
    ax.set_title(feat.replace('_', ' ').title(), fontweight='bold')
    ax.legend()

plt.suptitle('Feature Distributions: Churned vs Retained', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
FEATURES = [c for c in df.columns if c != 'label']
X = df[FEATURES]
y = df['label'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Compare three models
models = {
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, max_depth=5,
                                                        learning_rate=0.1, random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    auc  = roc_auc_score(y_test, y_prob)
    f1   = f1_score(y_test, y_pred)
    results[name] = {'model': model, 'auc': auc, 'f1': f1, 'proba': y_prob}
    print(f'{name:<25} AUC={auc:.4f}  F1={f1:.4f}')

In [ ]:
# ROC curves for all models
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for name, res in results.items():
    RocCurveDisplay.from_predictions(y_test, res['proba'], name=f"{name} (AUC={res['auc']:.3f})", ax=axes[0])
axes[0].set_title('ROC Curves', fontweight='bold')

# Confusion matrix for best model
best_name = max(results, key=lambda k: results[k]['auc'])
best_model = results[best_name]['model']
ConfusionMatrixDisplay.from_estimator(best_model, X_test, y_test, ax=axes[1], cmap='Blues')
axes[1].set_title(f'Confusion Matrix: {best_name}', fontweight='bold')

plt.tight_layout()
plt.savefig('churn_model.png', dpi=150)
plt.show()

In [ ]:
# Log best model to MLflow
mlflow.set_experiment('churn_prediction_notebook')
with mlflow.start_run(run_name=f'best_{best_name.replace(" ", "_")}'):
    mlflow.log_metric('roc_auc', results[best_name]['auc'])
    mlflow.log_metric('f1_score', results[best_name]['f1'])
    mlflow.log_param('model_type', best_name)
    mlflow.log_param('train_size', len(X_train))
    mlflow.sklearn.log_model(best_model, 'churn_model',
                              registered_model_name='churn_prediction_nb')
    print(f'Logged to MLflow: {best_name}, AUC={results[best_name]["auc"]:.4f}')

In [ ]:
# Feature importance
if hasattr(best_model, 'feature_importances_'):
    imp = pd.Series(best_model.feature_importances_, index=FEATURES).sort_values(ascending=True)
    plt.figure(figsize=(10, 6))
    imp.tail(12).plot(kind='barh', color='steelblue')
    plt.title(f'Top Feature Importances: {best_name}', fontweight='bold')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.show()

## Business Impact

- **Identify churners 2-4 weeks before they leave** → targeted win-back campaign
- **At 60% churn probability threshold**: send personalised discount email
- **Revenue saved**: If 10% of predicted churners are retained, and avg LTV = $300 → $30 per reactivation
- **Model tracked in MLflow** — version controlled, reproducible, deployable via FastAPI